# Laboratorio de regresión - 3

## Significancia de factores

|                |   |
:----------------|---|
| **Nombre**     |  JUAN PEDRO LEY VALDEZ |
| **Fecha**      |   01/02/2026|
| **Expediente** |   746385|

Descarga el archivo de publicidad y carga los datos (Advertising.csv).

In [5]:
import pandas as pd
import numpy as np
from scipy import stats

In [3]:
df = pd.read_csv('Advertising.csv', index_col=0)
print(df.head())

      TV  radio  newspaper  sales
1  230.1   37.8       69.2   22.1
2   44.5   39.3       45.1   10.4
3   17.2   45.9       69.3    9.3
4  151.5   41.3       58.5   18.5
5  180.8   10.8       58.4   12.9


**¿Hay alguna relación entre el presupuesto para publicidad y las ventas?**

Nuestra primera meta debe ser determinar si hay evidencia en los datos de que haya una asociación entre estas variables.

- ¿Por qué? ¿Qué resultaría si nos diéramos cuenta de la falta de relación entre el presupuesto de publicidad y las ventas?

In [6]:
X = df[['TV', 'radio', 'newspaper']].values
y = df['sales'].values
n = len(y)
p = X.shape[1]  # Número de predictores (3)

# 2. Agregar columna de intercepto (Unos)
X_b = np.c_[np.ones((n, 1)), X]

# 3. Calcular coeficientes (Betas) usando Ecuación Normal
# beta = (X^T * X)^-1 * X^T * y
beta = np.linalg.inv(X_b.T @ X_b) @ X_b.T @ y

# 4. Predicciones y Residuos
y_pred = X_b @ beta
residuals = y - y_pred
RSS = np.sum(residuals**2)
TSS = np.sum((y - np.mean(y))**2)

# 5. Calcular F-statistic
# F = ((TSS - RSS) / p) / (RSS / (n - p - 1))
F_statistic = ((TSS - RSS) / p) / (RSS / (n - p - 1))

# 6. Calcular p-value del F-statistic
p_value_F = 1 - stats.f.cdf(F_statistic, p, n - p - 1)

print(f"F-statistic: {F_statistic:.2f}")
print(f"p-value (F): {p_value_F:.2e}")

F-statistic: 570.27
p-value (F): 1.11e-16


### Respuesta:
Sí, existe una relación clara.

El **F-statistic** obtenido es de **570.27**, con un $p$-value prácticamente de cero ($< 10^{-16}$). Como el p-value es inferior al nivel de significancia (0.05), rechazamos la hipótesis nula ($H_0$) de que todos los coeficientes son cero. Esto confirma que al menos uno de los medios publicitarios está asociado significativamente con las ventas.

**¿Por qué es importante esto?**
Este es un test global de significancia. Si no lo hiciéramos y el $p$-value fuera alto, concluiríamos que **no hay relación lineal** entre el presupuesto y las ventas. En ese caso hipotético (falta de relación), el modelo no serviría para nada y cualquier análisis posterior de los coeficientes individuales (TV vs Radio) sería inválido.

**¿Qué tan fuerte es esta relación?**
Asumiendo que existe esta relación, ¿nos sirve conocer el impacto que tiene invertir en publicidad en las ventas?

In [7]:
# 1. RSE (Error Estándar Residual)
# RSE = sqrt(RSS / (n - p - 1))
RSE = np.sqrt(RSS / (n - p - 1))

# 2. R-squared (Coeficiente de determinación)
# R2 = 1 - (RSS / TSS)
R2 = 1 - (RSS / TSS)

# 3. Error porcentual respecto a la media de ventas
mean_sales = np.mean(y)
error_percentage = (RSE / mean_sales) * 100

print(f"RSE: {RSE:.4f}")
print(f"Promedio de Ventas: {mean_sales:.4f}")
print(f"Error porcentual: {error_percentage:.2f}%")
print(f"R-squared: {R2:.4f}")

RSE: 1.6855
Promedio de Ventas: 14.0225
Error porcentual: 12.02%
R-squared: 0.8972


### Respuesta:
La relación es bastante fuerte.

Para medirla utilizamos dos métricas:
1.  **RSE (Residual Standard Error):** Es **1.6855**. Esto significa que las ventas reales se desvían de la predicción del modelo en un promedio de 1,685 unidades. Dado que el promedio de ventas es 14,022, esto representa un **error porcentual de aproximadamente el 12%**. Aunque no es perfecto, es un error razonable para datos de marketing.
2.  **$R^2$ (R-cuadrada):** Es **0.8972**. Esto indica que cerca del **90% de la variabilidad en las ventas** es explicada por el presupuesto invertido en TV, Radio y Periódico.

**Conclusión:** Conocer la inversión en publicidad es muy útil, ya que nos permite explicar la gran mayoría de lo que ocurre con las ventas, dejando solo un 10% de variabilidad debida a otros factores no medidos.

**¿Cuáles medios están asociados con las ventas? ¿Qué tan grande es la asociación entre un medio específico y las ventas?**

Hay 3 medios distintos en los datos. ¿Sirve invertir en los 3? ¿Conviene más invertir sólo en uno?

**¿Qué tan seguros estamos de que podríamos predecir ventas futuras?**

**¿La relación es lineal?**

**¿Hay sinergia entre estos medios?**

Puede ser que gastar \\$50,000 en publicidad y otros \\$50,000 en radio es mejor opción que gastar \\$100,000 en televisión. A esto le llamamos efecto de interacción.

Podemos usar regresión lineal para responder todas estas preguntas.

Realiza una regresión lineal:

$$ \text{ventas} \approx \beta_0 + (\beta_1)(\text{TV})$$

In [8]:
from sklearn.linear_model import LinearRegression


X_tv = df[['TV']]
y = df['sales']

lm_tv = LinearRegression()
lm_tv.fit(X_tv, y)


b0 = lm_tv.intercept_
b1 = lm_tv.coef_[0]

print("Modelo estimado (TV):")
print(f"Ventas = {b0:.4f} + {b1:.4f} * TV")

Modelo estimado (TV):
Ventas = 7.0326 + 0.0475 * TV


### Verificando la precisión de nuestros coeficientes estimados

Recuerda que en el mundo real hay ruidos y errores de medición. Siempre se asume que la verdadera relación entre $X$ y $Y$ es $$Y = \beta_0 + \beta_1 X + \epsilon$$

Se asume que el término de error es independiente de $X$ (el error siempre es el mismo sin importar el valor de $X$). Este modelo describe a la *línea de regresión de la población*, que es la mejor aproximación de la verdadera relación entre $X$ y $Y$. Cuando usamos mínimos cuadrados encontramos la *línea de mínimos cuadrados*.

¿Cuál es la diferencia entre población y muestra?

### Respuesta:
La diferencia entre poblacion y muestra, es que poblacion se refiere al conjunto total de todos los posibles puntos de datos o individuos que cumplen con ciertas características. En el contexto de regresión, la población contiene la "verdad" absoluta sobre la relación entre X y y.

Muestra se refiere a un subconjunto de datos observado y recolectado de la población. Utilizamos la muestra para estimar los parámetros desconocidos de la población.

¿Cuál crees que sea la diferencia entre hacer una regresión con todos los datos de la población y una muestra de ella?

### Respuesta: 
La diferencia entre hacer una regresion con todos los datos y hacerla con una muestra, es que con acceso a todos los datos podriamos tener los verdaderos valores de $\beta_0$ y $\beta_1$. Esta línea es la mejor aproximación posible a la realidad. El único error existente sería el término de error irreducible $\epsilon$ (ruido aleatorio). Aquí no existen intervalos de confianza ni p-values, porque conocemos la verdad. Mientras que la regresion con la muestra es que como solo tenemos una fracción de los datos, calculamos estimaciones ($\hat{\beta}$). La línea de mínimos cuadrados intenta imitar a la línea poblacional, pero nunca será exactamente igual debido al sesgo de muestreo. Aquí sí necesitamos calcular errores estándar y p-values para saber qué tan confiables son nuestras estimaciones respecto a los verdaderos parámetros poblacionales.

La línea de regresión de la población no se puede observar. El concepto de comparar estas líneas es una extensión natural del acercamiento estadístico estándar de usar información de una muestra para estimar características de una población grande.

Imagina que quieres encontrar la altura promedio de un mexicano $\mu$. Medir a todos y cada uno de los mexicanos en situaciones similares, con la misma regla, mismo operador, y otras incontables formas de minimizar la variación de la medida es una tarea imposible. Lo que podemos asumir es que $\hat{\mu} = \bar{y}$. La media poblacional y la media muestral son diferentes, pero la media muestral es usualmente un buen estimado.

De la misma manera, como no contamos con el 100% de la información para hacer una regresión, los coeficientes $\beta_0$ y $\beta_1$ son desconocidos. Podemos estimarlos usando mínimos cuadrados, encontrando $\hat{\beta_0}$ y $\hat{\beta_1}$. Puede que las muestras que tengamos en ese momento estén un poco por encima de la media, pero otras muestras en otro momento puede que estén debajo de la media. En general, esperamos que el promedio de las aproximaciones $\hat{\mu}$ aproxime a $\mu$.

Esto lleva a la pregunta: ¿qué tan cercanos son nuestros coeficientes estimados a los verdaderos coeficientes? Utilizamos el concepto de error estándar para evaluar esto.

$$ \text{Var}(\hat{\mu})=\text{SE}(\hat{\mu})^2 = \frac{\sigma^2}{n} $$

Donde $\sigma$ es la desviación estándar de cada una de las observaciones $y_i$ de $Y$. El error estándar nos dice la cantidad promedio que el estimado difiere del valor verdadero. Podemos ver en la fórmula que entre más observaciones tengamos el error se hace más pequeño. Las fórmulas para errores estándar de $\hat{\beta_0}$ y $\hat{\beta_1}$ son:

$$ \text{SE}(\hat{\beta_0})^2 = \sigma^2 [\frac{1}{n} + \frac{\bar{x}^2}{\sum_{i=1}^n (x_i - \bar{x})^2}]$$

$$ \text{SE}(\hat{\beta_1})^2 = \frac{\sigma^2}{\sum_{i=1}^n (x_i - \bar{x})^2}$$

$$ \sigma^2 = \text{Var}(\epsilon) = \text{RSE}^2 = \frac{\text{RSS}}{n-p}$$

Para que estas fórmulas sean validas asumimos que los errores $\epsilon_i$ tienen varianza común $\sigma^2$ y que no están correlacionados.

Calcula los errores estándar de los coeficientes

In [10]:

# 1. Obtener predicciones y residuos usando el modelo anterior
y_pred = lm_tv.predict(X_tv)
residuals = y - y_pred

# 2. Calcular RSS y RSE
n = len(y)
RSS = np.sum(residuals**2)
RSE = np.sqrt(RSS / (n - 2)) # Grados de libertad: n - 2 (intercepto y pendiente)

# 3. Preparar la matriz X con intercepto (necesaria para la fórmula matricial)
# Tomamos los valores de X_tv y le pegamos una columna de unos a la izquierda
X_con_intercepto = np.column_stack([np.ones(n), X_tv.values])

# 4. Calcular la varianza de los coeficientes: (X^T * X)^-1 * RSE^2
# Esta es la fórmula clave del laboratorio
var_beta = np.linalg.inv(X_con_intercepto.T @ X_con_intercepto) * (RSE**2)

# 5. Obtener los Errores Estándar (raíz cuadrada de la diagonal)
std_errors = np.sqrt(np.diag(var_beta))

print(f"SE(Intercepto): {std_errors[0]:.4f}")
print(f"SE(TV): {std_errors[1]:.5f}")

SE(Intercepto): 0.4578
SE(TV): 0.00269


Estos errores se pueden usar para calcular intervalos de confianza. Un intervalo de confianza del $95\%$ se define como un rango de valores en el cuál se encuentra el desconocido valor verdadero con un $95\%$ de probabilidad.

Otra forma de verlo es que si tomamos muestras repetidas y construimos un intervalo de confianza para cada una, el $95\%$ de los intervalos creados van a contener el valor verdadero. Para la regresión el intervalo de confianza del $95\%$ toma la forma:

$$ \hat{\beta_j} \pm 2\text{SE}(\hat{\beta_j})$$

Calcula los intervalos de confianza para los coeficientes estimados:

In [13]:
from scipy import stats

# --- Cálculo de Intervalos de Confianza (95%) ---

# 1. Definir nivel de confianza y grados de libertad
confianza = 0.95
alpha = 1 - confianza
grados_libertad = n - 2


# Intervalo para el Intercepto
ci_b0_lower = b0 - 2 * std_errors[0]
ci_b0_upper = b0 + 2 * std_errors[0]

# Intervalo para TV
ci_b1_lower = b1 - 2 * std_errors[1]
ci_b1_upper = b1 + 2 * std_errors[1]


print(f"Intervalo de Confianza para Intercepto: [{ci_b0_lower:.4f}, {ci_b0_upper:.4f}]")
print(f"Intervalo de Confianza para TV:        [{ci_b1_lower:.4f}, {ci_b1_upper:.4f}]")

Intervalo de Confianza para Intercepto: [6.1169, 7.9483]
Intervalo de Confianza para TV:        [0.0422, 0.0529]


Los errores estándar también se usan para realizar pruebas de hipótesis. La prueba de hipótesis más común es probar la hipótesis nula de:

$$ H_0: \text{No hay relación entre } X \text{ y } Y \ \ \ \ (\beta_1=0)$$

contra la hipótesis alternativa:
$$ H_0: \text{Hay alguna relación entre } X \text{ y } Y \ \ \ (\beta_1 \neq 0)$$

Explica con tus palabras el significado de la hipótesis nula y la hipótesis alternativa.

### Respuesta
La hipótesis nula asume que no hay efecto o relación en una observación. Mientras que la alternativa propone que sí hay un efecto o relación. Se usan los datos para obtener pruebas suficientes para rechazar H0 y aceptar H1

Para probal la hipótesis nula debemos determinar si nuestro estimado $\hat{\beta_1}$ de $\beta_1$ está lo suficientemente alejado de cero para que podamos decir con confianza que este valor no es cero. 

¿Qué tan lejos? Depende de qué tanta confianza tengamos en el estimado encontrado. Si nuestro error estándar es pequeño y nuestro estimado está alejado de cero podríamos decir que hay muy poca probabilidad de que el valor verdadero sea 0. En cambio, si nuestro error estándar es grande y nuestro estimado está muy cerca de cero, entonces podrías ser que el valor verdadero sea cero y que no haya relación entre las variables.

Se calcula un *estadístico t* dado por
$$ t = \frac{\hat{\beta_j} - \mu}{\text{SE}(\hat{\beta_j})} $$

donde $\mu$ es el valor contra el que queremos probar.

Calcula el estadístico t para tus coeficientes estimados, usando como referencia la prueba de hipótesis.

In [15]:

t_stat_b0 = b0 / std_errors[0]
t_stat_tv = b1 / std_errors[1]

print(f"Estadístico t (Intercepto): {t_stat_b0:.4f}")
print(f"Estadístico t (TV):         {t_stat_tv:.4f}")

Estadístico t (Intercepto): 15.3603
Estadístico t (TV):         17.6676


La distribución t tiene forma de campana y se parece bastante a la distribución normal cuando $n > 30$. Ya sólo es cuestión de calcular la probabilidad de observar cualquier número tal que su valor absoluto sea igual o mayor que el valor absoluto del estadístico t calculado. En otras palabras:
$$ P(|x| \geq |t|) $$

A esta probabilidad la llamamos *p-value*. Un *p-value* pequeño indica que es poco probable que exista por puro azar una relación significativa entre predictor y respuesta, en caso de que no haya una asociación real entre predictor y respuesta. En otras palabras, el *p-value* te dice la probabilidad de que parezca que hay relación cuando no la hay.

Si el *p-value* es pequeño, inferimos que sí hay una asociación entre el predictor y la respuesta, y **rechazamos la hipótesis nula**.
  

¿Qué tan pequeño? Depende de la aplicación. Un valor muy común es del $5\%$.

Utiliza el siguiente código para calcular el *p-value* para tus coeficientes

`from scipy import stats`

`p_bj = 2*(1 - stats.t.cdf(np.abs(t_bj), n-p))`

In [17]:
from scipy import stats



# Definimos los grados de libertad (n - número de parámetros estimados)
# En regresión simple estimamos 2 parámetros: intercepto y pendiente
grados_libertad = n - 2

# Calculamos los p-values usando la distribución t acumulada (cdf)
# Multiplicamos por 2 porque es una prueba de dos colas (queremos saber si es diferente de 0, sea positivo o negativo)
p_value_b0 = 2 * (1 - stats.t.cdf(np.abs(t_stat_b0), df=grados_libertad))
p_value_tv = 2 * (1 - stats.t.cdf(np.abs(t_stat_tv), df=grados_libertad))

print(f"p-value (Intercepto): {p_value_b0:.4e}") # Notación científica para ver los ceros
print(f"p-value (TV):         {p_value_tv:.4e}")

p-value (Intercepto): 0.0000e+00
p-value (TV):         0.0000e+00


¿Se rechaza la hipótesis nula? ¿Qué significa?

### Respuesta
Dado que el p-value es mucho menor que cualquier nivel de significancia estándar ($\alpha = 0.05$ o $0.01$), **rechazamos la hipótesis nula**. Concluimos con seguridad estadística que existe una asociación significativa entre el presupuesto de publicidad en TV y las ventas.

Realiza otras dos regresiones. Ya tienes hecha la regresión de ventas dado el gasto en publicidad de TV. Realiza la regresión para gastos en radio y gastos en periódico. Organiza las respuestas para que debajo de esta celda se tenga:
- Título de regresión
- Coeficientes estimados
- Errores estándar de los coeficientes
- Intervalos de confianza
- Estadísticos t
- p-values
- Observaciones

In [19]:
# Radio 

df = pd.read_csv('Advertising.csv')
y = df['sales']
n = len(y)

# 1. Ajuste del Modelo (Radio)
X_radio = df[['radio']]
lm_radio = LinearRegression()
lm_radio.fit(X_radio, y)

# 2. Obtención de Estadísticos
b0 = lm_radio.intercept_
b1 = lm_radio.coef_[0]

# Errores Estándar (SE)
y_pred = lm_radio.predict(X_radio)
residuals = y - y_pred
RSS = np.sum(residuals**2)
RSE = np.sqrt(RSS / (n - 2))

X_design = np.column_stack([np.ones(n), X_radio.values])
var_beta = np.linalg.inv(X_design.T @ X_design) * (RSE**2)
std_errors = np.sqrt(np.diag(var_beta))

# Estadísticos t y p-values
t_stat_b0 = b0 / std_errors[0]
t_stat_b1 = b1 / std_errors[1]
p_val_b0 = 2 * (1 - stats.t.cdf(np.abs(t_stat_b0), n-2))
p_val_b1 = 2 * (1 - stats.t.cdf(np.abs(t_stat_b1), n-2))

# Intervalos de Confianza (aprox t=2)
ci_b0 = [b0 - 2*std_errors[0], b0 + 2*std_errors[0]]
ci_b1 = [b1 - 2*std_errors[1], b1 + 2*std_errors[1]]

# --- Salida Organizada ---
print(f"--- Regresión: Radio  ---")
print(f"Coeficientes Estimados:     b0 = {b0:.4f}, b1 = {b1:.4f}")
print(f"Errores Estándar (SE):      SE(b0) = {std_errors[0]:.4f}, SE(b1) = {std_errors[1]:.4f}")
print(f"Intervalos de Confianza:    b0: [{ci_b0[0]:.4f}, {ci_b0[1]:.4f}], b1: [{ci_b1[0]:.4f}, {ci_b1[1]:.4f}]")
print(f"Estadísticos t:             t(b0) = {t_stat_b0:.4f}, t(b1) = {t_stat_b1:.4f}")
print(f"p-values:                   p(b0) = {p_val_b0:.4e}, p(b1) = {p_val_b1:.4e}")

--- Regresión: Radio  ---
Coeficientes Estimados:     b0 = 9.3116, b1 = 0.2025
Errores Estándar (SE):      SE(b0) = 0.5629, SE(b1) = 0.0204
Intervalos de Confianza:    b0: [8.1858, 10.4374], b1: [0.1617, 0.2433]
Estadísticos t:             t(b0) = 16.5422, t(b1) = 9.9208
p-values:                   p(b0) = 0.0000e+00, p(b1) = 0.0000e+00


**Observaciones sobre la regresion de Radio:**

1.  **Relación Significativa:** El *p-value* para el coeficiente de Radio es prácticamente cero ($< 10^{-16}$), y el estadístico $t$ es de 9.92. Esto indica una relación muy fuerte y significativa entre el gasto en radio y las ventas.
2.  **Impacto:** El coeficiente es 0.2025. Esto significa que por cada unidad invertida en Radio, las ventas aumentan en promedio 0.20 unidades (mucho mayor que el 0.047 de TV).
3.  **Intervalo de Confianza:** El intervalo [0.16, 0.24] no incluye el cero, confirmando la asociación positiva.

In [20]:
# Periodico
# 1. Ajuste del Modelo (Periódico)
X_news = df[['newspaper']]
lm_news = LinearRegression()
lm_news.fit(X_news, y)

# 2. Obtención de Estadísticos
b0_n = lm_news.intercept_
b1_n = lm_news.coef_[0]

# Errores Estándar (SE)
y_pred_n = lm_news.predict(X_news)
residuals_n = y - y_pred_n
RSS_n = np.sum(residuals_n**2)
RSE_n = np.sqrt(RSS_n / (n - 2))

X_design_n = np.column_stack([np.ones(n), X_news.values])
var_beta_n = np.linalg.inv(X_design_n.T @ X_design_n) * (RSE_n**2)
std_errors_n = np.sqrt(np.diag(var_beta_n))

# Estadísticos t y p-values
t_stat_b0_n = b0_n / std_errors_n[0]
t_stat_b1_n = b1_n / std_errors_n[1]
p_val_b0_n = 2 * (1 - stats.t.cdf(np.abs(t_stat_b0_n), n-2))
p_val_b1_n = 2 * (1 - stats.t.cdf(np.abs(t_stat_b1_n), n-2))

# Intervalos de Confianza (aprox t=2)
ci_b0_n = [b0_n - 2*std_errors_n[0], b0_n + 2*std_errors_n[0]]
ci_b1_n = [b1_n - 2*std_errors_n[1], b1_n + 2*std_errors_n[1]]

# --- Salida Organizada ---
print(f"--- Regresión: Periódico  ---")
print(f"Coeficientes Estimados:     b0 = {b0_n:.4f}, b1 = {b1_n:.4f}")
print(f"Errores Estándar (SE):      SE(b0) = {std_errors_n[0]:.4f}, SE(b1) = {std_errors_n[1]:.4f}")
print(f"Intervalos de Confianza:    b0: [{ci_b0_n[0]:.4f}, {ci_b0_n[1]:.4f}], b1: [{ci_b1_n[0]:.4f}, {ci_b1_n[1]:.4f}]")
print(f"Estadísticos t:             t(b0) = {t_stat_b0_n:.4f}, t(b1) = {t_stat_b1_n:.4f}")
print(f"p-values:                   p(b0) = {p_val_b0_n:.4e}, p(b1) = {p_val_b1_n:.4e}")

--- Regresión: Periódico  ---
Coeficientes Estimados:     b0 = 12.3514, b1 = 0.0547
Errores Estándar (SE):      SE(b0) = 0.6214, SE(b1) = 0.0166
Intervalos de Confianza:    b0: [11.1086, 13.5942], b1: [0.0215, 0.0878]
Estadísticos t:             t(b0) = 19.8761, t(b1) = 3.2996
p-values:                   p(b0) = 0.0000e+00, p(b1) = 1.1482e-03


**Observaciones sobre la regresion con Periódico:**

1.  **Relación Débil pero Significativa:** El coeficiente de Periódico es 0.0547 con un *p-value* de 0.00115. Aunque es estadísticamente significativo (p < 0.05), la relación es mucho más débil comparada con TV o Radio.
2.  **Estadístico t:** El valor $t$ es 3.30. Es mayor que 2, por lo que rechazamos la hipótesis nula, pero es considerablemente menor que los valores de 17 (TV) y 9.9 (Radio) que vimos antes.
3.  **Intervalo de Confianza:** El intervalo [0.0215, 0.0878] es positivo pero se acerca más al cero.

## Regresión lineal múltiple

En lugar de hacer una regresión para cada factor independiente, quizás se puede extender el modelo para que tenga varios factores dentro:

$$ Y = \beta_0 + \beta_1 X_1 + \beta_2 X_2 + ... + \beta_p X_p + \epsilon $$

Para nuestro ejemplo de publicidad:

$$ \text{sales} = \beta_0 + \beta_1 (\text{TV}) + \beta_2 (\text{radio}) + \beta_3 (\text{newspaper}) + \epsilon $$

Utiliza la librería `statsmodels` para realizar la regresión. Por defecto la librería statsmodels no toma en cuenta el intercepto ($\beta_0$), por lo que se tendrá que agregar una columna de unos de tamaño *n* a la matriz X.

`import statsmodels.api as sm`

`ols = sm.OLS(Y, X)`

`results = ols.fit()`

`results.summary()`

In [22]:
import statsmodels.api as sm


X = df[['TV', 'radio', 'newspaper']]
y = df['sales']


X = sm.add_constant(X)

model = sm.OLS(y, X)
results = model.fit()

print(results.summary())

                            OLS Regression Results                            
Dep. Variable:                  sales   R-squared:                       0.897
Model:                            OLS   Adj. R-squared:                  0.896
Method:                 Least Squares   F-statistic:                     570.3
Date:                Sun, 01 Feb 2026   Prob (F-statistic):           1.58e-96
Time:                        21:35:00   Log-Likelihood:                -386.18
No. Observations:                 200   AIC:                             780.4
Df Residuals:                     196   BIC:                             793.6
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          2.9389      0.312      9.422      0.0

¿Qué diferencias puedes observar entre los *p-values* de una regresión múltiple y los encontrados en las regresiones simples? ¿Por qué crees que existen estas diferencias?

### Respuesta
Las diferencias que podemos notar, es que cuando hacemos la regresión múltiple, el periódico deja de ser significativo. Esto sugiere que la relación que vimos antes en la regresión simple del periódico era engañosa posiblemente correlacionada con la radio, pero no causante de ventas por sí misma.

In [24]:
print(df.corr())

            Unnamed: 0        TV     radio  newspaper     sales
Unnamed: 0    1.000000  0.017715 -0.110680  -0.154944 -0.051616
TV            0.017715  1.000000  0.054809   0.056648  0.782224
radio        -0.110680  0.054809  1.000000   0.354104  0.576223
newspaper    -0.154944  0.056648  0.354104   1.000000  0.228299
sales        -0.051616  0.782224  0.576223   0.228299  1.000000


## Referencia

James, G., Witten, D., Hastie, T., Tibshirani, R.,, Taylor, J. (2023). An Introduction to Statistical Learning with Applications in Python. Cham: Springer. ISBN: 978-3-031-38746-3

## Addendum

Para calcular los *p-values* de los parámetros sin `statsmodels`:
1. Calcular RSS
2. Calcular RSE
3. `var_beta = np.linalg.inv(X.T @ X) * rse**2` (X con columna de intercepto)
4. `std_beta = np.sqrt(var_beta.diagonal())` El orden de los valores corresponde al orden de los factores en las columnas de la matriz $X$.
5. Calcular *estadístico t*
6. Calcular *p-value*